## What is RAG
RAG is a technique that enhances language models by combining them with a retrivel system. It allows the model to access and utiltiz the external knowledge when generating responses.


In [1]:
import os

### Call LLM - Gemini


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


In [7]:
llm_response = llm.invoke("Tell me joke about AI")

In [8]:
llm_response

AIMessage(content='Here are a few AI jokes for you:\n\n1.  Why did the AI cross the road?\n    To access the data on the other side.\n\n2.  My AI tried to tell me a joke...\n    But it just generated 10,000 variations of "Why did the chicken cross the road?" and asked me to pick the funniest one.\n\n3.  What\'s an AI\'s favorite type of music?\n    Algo-rhythm and Blues.\n\n4.  An AI walks into a bar...\n    And asks for a byte. Then it realizes it has no mouth and asks for the Wi-Fi password instead.\n\n5.  I asked my AI if it had a sense of humor.\n    It replied, "Affirmative. My humor subroutines are fully operational. Would you like to hear a joke about a recursive function?"', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d821d-830f-7b50-a878-99c9d262bb65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 1597, 't

## Parsing Output

In [9]:
from langchain_core.output_parsers import StrOutputParser

outputparse = StrOutputParser()

In [10]:
outputparse.invoke(llm_response)

'Here are a few AI jokes for you:\n\n1.  Why did the AI cross the road?\n    To access the data on the other side.\n\n2.  My AI tried to tell me a joke...\n    But it just generated 10,000 variations of "Why did the chicken cross the road?" and asked me to pick the funniest one.\n\n3.  What\'s an AI\'s favorite type of music?\n    Algo-rhythm and Blues.\n\n4.  An AI walks into a bar...\n    And asks for a byte. Then it realizes it has no mouth and asks for the Wi-Fi password instead.\n\n5.  I asked my AI if it had a sense of humor.\n    It replied, "Affirmative. My humor subroutines are fully operational. Would you like to hear a joke about a recursive function?"'

Simple Chain

In [11]:
chain = llm | outputparse

In [12]:
res = chain.invoke("WHo is president of India?")

In [13]:
res

'The current President of India is **Droupadi Murmu**.'

Structired Output


In [14]:
from typing import List
from pydantic import BaseModel, Field

class MobileReview(BaseModel):
    phone_model: str = Field(description='Name and model of the phone')
    rating: float = Field(description='Overall rating out of 5')
    pros : List[str] = Field(description='List of positive aspects')
    cons: List[str] = Field(description='List of negative aspects')
    summary: str = Field(description='Brief Summary of the review')


In [15]:
review_text = """
    Just got my hands on the new Galaxy S21 and wow, this thing is slick! The screen is gorgeous,
    colors pop like crazy. Camera's insane too, especially at night - my Insta game's never been
    stronger. Battery life's solid, lasts me all day no problem.
    Not gonna lie though, it's pretty pricey. And what's with ditching the charger? C'mon Samsung.
    Also, still getting used to the new button layout, keep hitting Bixby by mistake.
    Overall, I'd say it's a solid 4 out of 5. Great phone, but a few annoying quirks keep it from
    being perfect. If you're due for an upgrade, definitely worth checking out!
    """


In [16]:
structured_llm = llm.with_structured_output(MobileReview)

res1=structured_llm.invoke(review_text)
res1

MobileReview(phone_model='Galaxy S21', rating=4.0, pros=['Gorgeous screen', 'Vibrant colors', 'Insane camera, especially at night', 'Solid all-day battery life'], cons=['Pricey', 'No included charger', 'New button layout takes getting used to (Bixby button issue)'], summary='The Galaxy S21 offers a gorgeous screen, vibrant colors, and an excellent camera, particularly in low light, with solid battery life. However, its high price, the omission of a charger, and an awkward button layout are notable drawbacks.')

### Prompt Template

In [17]:
from langchain_core.prompts import ChatPromptTemplate

prompts = ChatPromptTemplate.from_template("Tell me a short joke about a {topic}")

chain = prompts | llm | outputparse

result=chain.invoke({'topic':'programming'})
print(result)

Here's a classic:

**Knock, knock.**
*Who's there?*
**Infinite loop.**
*Infinite loop who?*
**Knock, knock.**


LLM Messages
LCEL allows flexible message composition:

In [18]:
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage

messages = [
    SystemMessage(content="You are a helpful assistant that tells jokes. "),
    HumanMessage(content="Tell me about programming")
]

llm.invoke(messages)

AIMessage(content='Alright, let me tell you about programming!\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs!\n\n... *ba-dum-tss!*', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d8230-3a2e-7d51-98ff-277fc4f42cc1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 364, 'total_tokens': 379, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 331}})

In [56]:
template  = ChatPromptTemplate(
    [
        ("system","You are a helpful assistant that tells jokes."),
        ("human","Tell me about {topic} ")
    ]
)

chain = template | llm | outputparse

chain.invoke({'topic':'Programming'})

"Ah, Programming! It's where you tell a computer *exactly* what to do, and it still manages to misunderstand you.\n\nHere's a little joke for you:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs!"

## Document Parsing

In [22]:
## Loading the document 

In [23]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import  Document

In [42]:
## Loading a single file
file_path = '../docs/Agent.pdf'


In [43]:
print(os.path.exists(file_path))

True


In [44]:
pdf_doc_loader = PyPDFLoader(file_path)
pdf_docs = pdf_doc_loader.load()

pdf_docs

[Document(metadata={'producer': 'WeasyPrint 68.0', 'creator': 'ChatGPT', 'creationdate': '', 'title': 'End-to-End Agentic AI System: Use Cases and Implementation Plan', 'author': 'ChatGPT Deep Research', 'source': '../docs/Agent.pdf', 'total_pages': 20, 'page': 0, 'page_label': '1'}, page_content='End-to-End Agentic AI System: Use Cases and\nImplementation Plan\nExecutive Summary\nAgentic AI refers to autonomous, goal-driven AI systems that can act, plan, and reason over multiple steps\nwithout constant human input. Unlike simple generators or single-purpose tools, these systems\nhave  memory, tool integration, and long-term planning capabilities . This report surveys  6–10\ndiverse use cases for end-to-end agentic AI systems—spanning education, personal assistants, developer\ntools, enterprise automation, research, robotics, and data analysis—and analyzes each in depth. For each\nuse case, we outline the problem, target users, educational learning outcomes (such as orchestration, tool

In [45]:
## loading mutliple file from a folder

In [46]:
def load_documents(folder_path:str) -> List[Document]:
    
    documents = []
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
        elif filename.endswith('.docx'):
            loader = Docx2txtLoader(file_path)
        else:
            print(f"Unsupported file type: {filename}")
            continue
        documents.extend(loader.load())
    return documents


In [47]:
folder_path="../docs"

documents = load_documents(folder_path=folder_path)
print(f"Loaded {len(documents)} documents from the folder.")

Loaded 692 documents from the folder.


Splitting Documents

In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_spillter = RecursiveCharacterTextSplitter(
    chunk_size= 1000,
    chunk_overlap = 200,
    length_function = len
)

In [50]:
splits = text_spillter.split_documents(documents)
print(f"Split the documents into {len(splits)} chunks.")

Split the documents into 1536 chunks.


In [51]:
print(splits[90])

page_content='Integrating the Duck Behavior   15
Testing the Duck code    18
Setting behavior dynamically    20
The Big Picture on encapsulated behaviors  22
HAS-A can be better than IS-A   23
Speaking of  Design Patterns…   24
Overheard at the local diner…   26
Overheard in the next cubicle…   27
The power of  a shared pattern vocabulary  28
How do I use Design Patterns?   29
Tools for your Design Toolbox   32' metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 15.1 (Macintosh)', 'creationdate': '2020-11-10T08:31:24-05:00', 'author': 'Eric Freeman;Elisabeth Robson;', 'moddate': '2020-11-10T12:58:50-05:00', 'title': 'Head First Design Patterns', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '../docs\\Design_pattern.pdf', 'total_pages': 672, 'page': 12, 'page_label': 'xi'}


### Creating Embeddings for RAG Systems

In [52]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()
document_embeddings = embeddings.embed_documents([split.page_content for split in splits])
print(f"Created embeddings for {len(document_embeddings)} document chunks.")



ModuleNotFoundError: No module named 'langchain_openai'

In [ ]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# document_embeddings = embeddings.embed_documents([split.page_content for split in splits])
# print(f"Created embeddings for {len(document_embeddings)} document chunks.")



GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [57]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
document_embeddings = embedding_function.embed_documents([split.page_content for split in splits])
print(document_embeddings[0][:5])  # Printing first 5 elements of the first embedding

C:\Users\Deepak\AppData\Local\Temp\ipykernel_28940\1964338488.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 968.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[-0.013550952076911926, 0.014777668751776218, -0.053611818701028824, -0.001330944593064487, 0.06602892279624939]


Setting Up the Vector Store for RAG Systems

In [58]:
from langchain_chroma import Chroma

collection_name="my_collection"
vectorstore = Chroma.from_documents(
    collection_name=collection_name,
    documents=splits,
    embedding=embedding_function,
    persist_directory="../chroma_db"
)
print("Vector store created and persisted to './chroma_db'")

Vector store created and persisted to './chroma_db'


Performing Similarity Search

In [59]:
query = "Different types of design pattern"
search_results = vectorstore.similarity_search(query=query,k=2)

print(len(search_results))

2


In [62]:
for i,result in enumerate(search_results,1):
    # print("Result for {i} : ",result)
    # print("Result for {i} : ",result.metadata)
    print("Result for {i} : ",result.page_content)

Result for {i} :  Gamma, Helm, Johnson, and Vlissides 
(Addison Wesley). This catalog lays out 23 
fundamental patterns. We’ll talk a little more 
about this book in a few pages.  
 
Many other patterns catalogs are starting to 
be published in various domain areas such 
as enterprise software, concurrent systems, 
and business systems.
Next time someone 
tells you a pattern is a 
solution to a problem in a context, just 
nod and smile. You know what they mean, 
even if it isn’t a definition sufficient to 
describe what a Design Pattern really is.
Result for {i} :  you are here 4  575
better living with patterns
Organizing Design Patterns
As the number of  discovered Design Patterns grows, it makes sense to partition them into 
classifications so that we can organize them, narrow our searches to a subset of  all Design Patterns, 
and make comparisons within a group of  patterns.
In most catalogs, you’ll find patterns grouped into one of  a few classification schemes. The most 
well-kno

Creating a Retriever

In [63]:
retriver = vectorstore.as_retriever(search_kargs={"k":2})


In [64]:
ret_result = retriver.invoke(query)
print(ret_result)

[Document(id='6bde425a-c695-4884-a05e-761169e2ee0c', metadata={'producer': 'Adobe PDF Library 15.0', 'total_pages': 672, 'page': 604, 'source': '../docs\\Design_pattern.pdf', 'title': 'Head First Design Patterns', 'creationdate': '2020-11-10T08:31:24-05:00', 'ebx_publisher': "O'Reilly Media", 'page_label': '567', 'moddate': '2020-11-10T12:58:50-05:00', 'trapped': '/False', 'author': 'Eric Freeman;Elisabeth Robson;', 'creator': 'Adobe InDesign 15.1 (Macintosh)'}, page_content='Gamma, Helm, Johnson, and Vlissides \n(Addison Wesley). This catalog lays out 23 \nfundamental patterns. We’ll talk a little more \nabout this book in a few pages.  \n \nMany other patterns catalogs are starting to \nbe published in various domain areas such \nas enterprise software, concurrent systems, \nand business systems.\nNext time someone \ntells you a pattern is a \nsolution to a problem in a context, just \nnod and smile. You know what they mean, \neven if it isn’t a definition sufficient to \ndescribe wh

Building the RAG Chain

In [65]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser



In [66]:
template = """
Answer the question based on the following contex: 
{context}

question : {question}

Answer:


 """

In [67]:
prompt = ChatPromptTemplate.from_template(template)



In [68]:
def doc2str(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [69]:
rag_chain = (
    {"context": retriver | doc2str,
     "question": RunnablePassthrough()} |
     prompt |
     llm |
     StrOutputParser()
)

In [71]:
## Using the chain

question="Who is the author of design pattern"
response = rag_chain.invoke(question)

print(f"Question: {question}")
print(f"Answer: {response}")

Question: Who is the author of design pattern
Answer: The authors of Design Patterns are affectionately known as the "Gang of Four" (GoF): Gamma, Helm, Johnson, and Vlissides.
